<a href="https://colab.research.google.com/github/AnastasiyaOvechko/Digidrat/blob/main/PD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from openpyxl import load_workbook

# ====================================================
# НАСТРОЙКИ
# ====================================================

FILE_SRC = "PD RMC 01.xlsx"
FILE_DST = "Pd_01 01 2026.xlsx"

SRC_INPUT_SHEET = "Аркуш1"
SRC_OUTPUT_SHEET = "Аркуш2"

DST_MAIN_SHEET = "PDm"
DST_BAL_SHEET = "бал нов"

DATE_LIMIT = pd.to_datetime("2024-06-01")

COMMENT_TEXT = "Обробка виконана автоматично згідно методички (п.19)."

# ====================================================
# ЧАСТЬ 1. PD RMC 01.xlsx
# ====================================================

# --- чтение Аркуш1 ---
df = pd.read_excel(
    FILE_SRC,
    sheet_name=SRC_INPUT_SHEET,
    header=2
)

# удаляем МКК и клиент
if "Тип отчетности" in df.columns:
    df = df[~df["Тип отчетности"].isin(["Mkk", "Client"])]


# Дата установки статуса → без времени
STATUS_COL = "Дата установки статуса"
MOD_STATUS_COL = "Дата установки статуса_Модифікована"

df[STATUS_COL] = pd.to_datetime(df[STATUS_COL], errors="coerce")
df[MOD_STATUS_COL] = df[STATUS_COL].dt.date

# сортировка по модифицированной дате
df = df.sort_values(
    by=[
        "ОКПО",
        "Дата 1",
        "Дата установки статуса_Модифікована"
    ],
    ascending=[
        True,   # ОКПО — по возрастанию
        False,  # Дата 1 — от новой к старой
        False   # Дата установки статуса_Модифікована — от новой к старой
    ]
)


# проверка
if "ОКПО" in df.columns:
    _ = (
        df.sort_values(["ОКПО", MOD_STATUS_COL], ascending=[True, False])
        .groupby("ОКПО")[MOD_STATUS_COL]
        .apply(lambda x: x.is_monotonic_decreasing)
    )

# фильтр по Дата 1 и добавление столбцов
df["Дата 1"] = pd.to_datetime(df["Дата 1"], errors="coerce")
df = df[df["Дата 1"] >= DATE_LIMIT]

OKPO_COL = "ОКПО"
DATE1_COL = "Дата 1"

# оставляем ТОЛЬКО первую строку по ОКПО
df = df.drop_duplicates(
    subset=OKPO_COL,
    keep="first"
)

# добавляем два столбца после "Дата 1"
idx = df.columns.get_loc(DATE1_COL)
df.insert(idx + 1, "flag_1", 1)
df.insert(idx + 2, "flag_filter", 1)


# ---------- Аркуш2 ----------
with pd.ExcelWriter(
    FILE_SRC,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    df.to_excel(writer, sheet_name=SRC_OUTPUT_SHEET, index=False)

wb = load_workbook(FILE_SRC)
ws = wb[SRC_OUTPUT_SHEET]
ws["S2"] = COMMENT_TEXT
wb.save(FILE_SRC)

# ====================================================
# ЧАСТЬ 2. Pd_01 01 2026.xlsx
# ====================================================

df_dst = pd.read_excel(FILE_DST, sheet_name=DST_MAIN_SHEET)

# перенос ключевых данных
BASE_COLS = [
    "UID заявки",
    "Статус",
    "ОКПО",
    "Клиент",
    MOD_STATUS_COL,
    "Дата 1"
]

df_dst[BASE_COLS] = df[BASE_COLS].values

# qq1–qq12
qq_cols = [c for c in df.columns if c.lower().startswith("qq")]
df_dst[qq_cols] = df[qq_cols].values

# sq → Yes / No → 1 / 0
sq_cols = [c for c in df_dst.columns if c.lower().startswith("sq")]
df_dst[sq_cols] = df_dst[sq_cols].replace({"Yes": 1, "No": 0})

# ---------- запись PDm ----------
with pd.ExcelWriter(
    FILE_DST,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    df_dst.to_excel(writer, sheet_name=DST_MAIN_SHEET, index=False)

BAL_COLS = ["ОКПО", "Клиент", "Дата 1", "UID заявки", "Балл"]
STOP_COLS = [c for c in df_dst.columns if c.lower().startswith("sq")]

df_bal = df_dst[BAL_COLS + STOP_COLS]

with pd.ExcelWriter(
    FILE_DST,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    df_bal.to_excel(writer, sheet_name=DST_BAL_SHEET, index=False)

print("✅ Обробка завершена.")
